# 🌊 Simulate Water Quality Drift

## Purpose
Demo utility: Generates shifted inference data to test drift detection system

## Drift Types
1. **Feature Shift**: Shifts water quality parameter means by configurable standard deviations
2. **Label Noise**: Randomly flips a fraction of ground truth labels
3. **Prediction Drift**: Changes model prediction distribution

## Steps
1. Load current inference data
2. Generate drift scenarios (chemical contamination, seasonal changes, etc.)
3. Apply water quality specific drift patterns
4. Append shifted data to inference table
5. Trigger monitor refresh for detection

## Notes
- Demo purpose only - not for production use
- Simulates real-world water quality drift scenarios
- Tests monitoring and alerting system effectiveness

In [ ]:
# 📦 Install required packages
%pip install pandas numpy databricks-sdk

# Restart Python 
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries
import pandas as pd
import numpy as np
import time
from databricks.sdk import WorkspaceClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

# Initialize
spark = SparkSession.builder.getOrCreate()
w = WorkspaceClient()

print("✅ Libraries loaded for drift simulation")

In [ ]:
# 🎯 Configuration - Get from Bundle Parameters
catalog_name = spark.conf.get("catalog_name", "dev_digital_engineering_services")
schema_name = spark.conf.get("schema_name", "default")
drift_type = spark.conf.get("drift_type", "feature_shift")  # feature_shift, label_noise, both
drift_magnitude = float(spark.conf.get("drift_magnitude", "2.0"))  # Standard deviations

# Table names
INFERENCE_TABLE = f"{catalog_name}.{schema_name}.water_quality_inferences"

# Drift scenarios
DRIFT_SCENARIOS = {
    "chemical_contamination": {
        "description": "Chemical contamination affecting multiple parameters",
        "affected_features": ["Chloramines", "Trihalomethanes", "Organic_carbon"],
        "shift_direction": [2.0, 2.5, 1.8]  # Higher values = contamination
    },
    "seasonal_change": {
        "description": "Seasonal water source changes", 
        "affected_features": ["ph", "Hardness", "Turbidity"],
        "shift_direction": [-1.5, 1.2, 1.0]  # pH lower, hardness higher, turbidity higher
    },
    "equipment_malfunction": {
        "description": "Water treatment equipment malfunction",
        "affected_features": ["Sulfate", "Conductivity", "Solids"],
        "shift_direction": [1.8, 2.2, 1.5]  # Poor treatment = higher values
    },
    "source_change": {
        "description": "Change in water source (well to surface water)",
        "affected_features": ["Hardness", "Solids", "Organic_carbon", "Turbidity"],
        "shift_direction": [-2.0, -1.5, 2.5, 2.0]  # Softer but more organic/turbid
    }
}

print(f"🎯 Drift Simulation Configuration:")
print(f"   📊 Target Table: {INFERENCE_TABLE}")
print(f"   🌊 Drift Type: {drift_type}")
print(f"   📈 Magnitude: {drift_magnitude} std devs")

In [ ]:
# 📊 Load Current Inference Data
print("📊 Loading current inference data for drift simulation...")

# Load recent inference data
current_data = spark.sql(f"""
    SELECT * FROM {INFERENCE_TABLE}
    WHERE DATE(prediction_timestamp) >= CURRENT_DATE() - INTERVAL 1 DAY
    ORDER BY prediction_timestamp DESC
    LIMIT 50
""").toPandas()

if len(current_data) == 0:
    print("❌ No recent inference data found")
    print("💡 Run batch inference job first to generate data")
    raise Exception("No data available for drift simulation")

print(f"✅ Loaded {len(current_data)} recent inference records")

# Show current data statistics
print(f"📈 Current Data Statistics:")
water_features = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 
                 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

current_stats = current_data[water_features].describe()
print(current_stats.round(2))

In [ ]:
# 🎯 Select Drift Scenario
print("🎯 Available drift scenarios:")
for i, (scenario_name, scenario) in enumerate(DRIFT_SCENARIOS.items()):
    print(f"   {i+1}. {scenario_name}: {scenario['description']}")

# For demo, let's use chemical contamination scenario
selected_scenario = "chemical_contamination"
scenario_config = DRIFT_SCENARIOS[selected_scenario]

print(f"\n🌊 Selected Scenario: {selected_scenario}")
print(f"📋 Description: {scenario_config['description']}")
print(f"🎯 Affected Features: {scenario_config['affected_features']}")
print(f"📈 Shift Directions: {scenario_config['shift_direction']}")

In [ ]:
# 🌊 Generate Drift Data
print("🌊 Generating drift data...")

# Create drifted version of current data
drifted_data = current_data.copy()

# Apply feature drift based on scenario
if drift_type in ["feature_shift", "both"]:
    print(f"🔧 Applying feature shift...")
    
    affected_features = scenario_config['affected_features']
    shift_directions = scenario_config['shift_direction']
    
    for feature, shift_magnitude in zip(affected_features, shift_directions):
        if feature in drifted_data.columns:
            # Calculate feature statistics
            feature_mean = current_data[feature].mean()
            feature_std = current_data[feature].std()
            
            # Apply drift (shift by magnitude * std in specified direction)
            shift_amount = shift_magnitude * feature_std * drift_magnitude
            drifted_data[feature] = drifted_data[feature] + shift_amount
            
            # Ensure values stay positive (water quality can't be negative)
            drifted_data[feature] = np.maximum(drifted_data[feature], 0.1)
            
            print(f"   ✅ {feature}: shifted by {shift_amount:.2f} units")
            print(f"      Original mean: {feature_mean:.2f} → New mean: {drifted_data[feature].mean():.2f}")

# Re-engineer features with drifted values
print(f"🔧 Re-engineering features...")
drifted_data['ph_normalized'] = drifted_data['ph'] / 14
drifted_data['hardness_ratio'] = drifted_data['Hardness'] / (drifted_data['Solids'] + 1)
drifted_data['organic_load'] = drifted_data['Organic_carbon'] * drifted_data['Chloramines']
drifted_data['water_quality_index'] = (
    drifted_data['ph_normalized'] + 
    (1 / (drifted_data['Turbidity'] + 1)) + 
    (1 / (drifted_data['hardness_ratio'] + 1))
) / 3

print(f"✅ Feature engineering updated for drifted data")

In [ ]:
# 🎲 Apply Label Noise (if configured)
if drift_type in ["label_noise", "both"]:
    print(f"🎲 Applying label noise...")
    
    # Flip fraction of labels to simulate ground truth changes
    noise_fraction = 0.15  # 15% label noise
    n_samples = len(drifted_data)
    n_flips = int(n_samples * noise_fraction)
    
    # Randomly select samples to flip
    flip_indices = np.random.choice(n_samples, n_flips, replace=False)
    
    if 'Potability' in drifted_data.columns:
        original_labels = drifted_data.loc[flip_indices, 'Potability'].copy()
        drifted_data.loc[flip_indices, 'Potability'] = 1 - drifted_data.loc[flip_indices, 'Potability']
        
        print(f"   🔄 Flipped {n_flips} labels ({noise_fraction*100:.1f}%)")
        print(f"   📊 Label distribution before: {pd.Series(original_labels).value_counts().to_dict()}")
        print(f"   📊 Label distribution after: {drifted_data['Potability'].value_counts().to_dict()}")
    else:
        print(f"   ⚠️ No Potability column found - skipping label noise")

print(f"✅ Drift simulation completed")

In [ ]:
# 🚀 Re-run Predictions with Drifted Data  
print("🚀 Re-running predictions on drifted data...")

# Simulate prediction drift by adding noise to prediction confidence
# In real scenarios, this happens naturally when features drift

# Add uncertainty to predictions based on drift magnitude
confidence_noise = np.random.normal(0, drift_magnitude * 0.05, len(drifted_data))
drifted_data['prediction_confidence'] = np.clip(
    drifted_data['prediction_confidence'] + confidence_noise, 
    0.1, 0.99
)

# Occasionally flip predictions for heavily drifted samples
high_drift_mask = abs(confidence_noise) > drift_magnitude * 0.08
n_flipped_preds = high_drift_mask.sum()

if n_flipped_preds > 0:
    drifted_data.loc[high_drift_mask, 'predicted_potability'] = (
        1 - drifted_data.loc[high_drift_mask, 'predicted_potability']
    )
    print(f"🔄 Flipped {n_flipped_preds} predictions due to high drift")

# Update probabilities to be consistent  
drifted_data['potable_probability'] = np.where(
    drifted_data['predicted_potability'] == 1,
    drifted_data['prediction_confidence'],
    1 - drifted_data['prediction_confidence']
)
drifted_data['not_potable_probability'] = 1 - drifted_data['potable_probability']

print(f"✅ Prediction drift applied")

# Show drift summary
print(f"\n📊 DRIFT SUMMARY:")
print(f"   Scenario: {selected_scenario}")
print(f"   Samples Generated: {len(drifted_data)}")
if n_flipped_preds > 0:
    print(f"   Predictions Changed: {n_flipped_preds}")
print(f"   Avg Confidence Change: {(drifted_data['prediction_confidence'] - current_data['prediction_confidence']).mean():.3f}")

In [ ]:
# 💾 Append Drifted Data to Inference Table
print("💾 Appending drifted data to inference table...")

# Update timestamps to make it recent
drifted_data['prediction_timestamp'] = pd.Timestamp.now()

# Create new inference IDs
max_inference_id = current_data['inference_id'].max() if 'inference_id' in current_data.columns else 0
drifted_data['inference_id'] = range(max_inference_id + 1, max_inference_id + 1 + len(drifted_data))

# Add drift simulation metadata
drifted_data['drift_simulation'] = True
drifted_data['drift_scenario'] = selected_scenario
drifted_data['drift_magnitude'] = drift_magnitude

# Convert to Spark DataFrame and append
drifted_spark_df = spark.createDataFrame(drifted_data)

drifted_spark_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(INFERENCE_TABLE)

print(f"✅ {len(drifted_data)} drifted samples appended to {INFERENCE_TABLE}")

# Verify the append
total_records = spark.sql(f"SELECT COUNT(*) FROM {INFERENCE_TABLE}").collect()[0][0]
drift_records = spark.sql(f"SELECT COUNT(*) FROM {INFERENCE_TABLE} WHERE drift_simulation = true").collect()[0][0]

print(f"📊 Table Status:")
print(f"   Total Records: {total_records:,}")
print(f"   Drift Records: {drift_records:,}")
print(f"   Normal Records: {total_records - drift_records:,}")

In [ ]:
# 🚨 Trigger Monitor Refresh
print("🚨 Triggering monitor refresh for drift detection...")

try:
    # Refresh the monitor to analyze new data
    w.quality_monitors.run_refresh(INFERENCE_TABLE)
    print(f"✅ Monitor refresh triggered")
    print(f"⏳ Monitor will analyze new data and detect drift")
    
    # Wait a moment for processing to start
    time.sleep(10)
    
    print(f"📊 Monitor Status:")
    monitor_info = w.quality_monitors.get(INFERENCE_TABLE)
    print(f"   Status: {monitor_info.status}")
    
except Exception as e:
    print(f"⚠️ Monitor refresh error: {e}")
    print(f"💡 Monitor may need manual refresh in Databricks UI")

print(f"\n✅ DRIFT SIMULATION COMPLETED!")
print(f"🌊 Scenario: {selected_scenario} ({scenario_config['description']})")
print(f"📈 Magnitude: {drift_magnitude} standard deviations")
print(f"📊 Samples: {len(drifted_data)} drifted records added")
print(f"🚨 Monitor: Refresh triggered for drift analysis")

print(f"\n🎯 NEXT STEPS:")
print(f"1. Wait 5-10 minutes for monitor to process new data")
print(f"2. Run drift_detection notebook to analyze results")
print(f"3. Check monitoring dashboard for drift alerts")
print(f"4. Verify email notifications (if configured)")

# Show comparison stats
print(f"\n📊 DRIFT COMPARISON:")
for feature in scenario_config['affected_features'][:3]:  # Show top 3
    if feature in current_data.columns:
        original_mean = current_data[feature].mean()
        drifted_mean = drifted_data[feature].mean()
        change_pct = ((drifted_mean - original_mean) / original_mean) * 100
        print(f"   {feature}: {original_mean:.2f} → {drifted_mean:.2f} ({change_pct:+.1f}%)")